
# 🧠 우울감 × 좌식시간 × 수면 × 복부비만 분석  
**데이터:** CDC NHANES 2013–2014 (Kaggle 미러)  
**핵심 질문:** 우울감이 복부비만과 관련이 있다면, 그 경로는 `운동 부족`보다 `오래 앉아있기`와 `수면`으로 더 잘 설명되는가?

---

## 이 노트북의 포인트
- PHQ-9 (`DPQ010`–`DPQ090`)로 우울감 측정
- 좌식시간 (`PAD680`) + 수면시간 (`SLD010H`)을 핵심 매개/설명 변수로 사용
- BMI보다 **중심성 비만** 지표인 **Waist-to-Height Ratio (WHtR)** 를 1차 결과로 사용
- NHANES 2-year MEC weight(`WTMEC2YR`)를 반영한 가중 회귀분석
- GitHub/Colab 친화형 구조

> 주의: Python `statsmodels`에서는 NHANES의 복합표본설계(PSU/strata)를 완전하게 재현하지 못합니다.  
> 이 노트북은 **가중치 반영 + robust SE 기반의 실용형 분석**입니다. 최종 투고 전에는 R `survey` 패키지 재현을 권장합니다.


## STEP 0. Kaggle 인증

In [ ]:

from google.colab import files
uploaded = files.upload()  # kaggle.json 선택

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle 인증 완료')


## STEP 1. 환경 설정 + 데이터 다운로드

In [ ]:

!pip install -q kaggle pingouin statsmodels seaborn

import warnings, os, glob, math, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pingouin as pg

import matplotlib.font_manager as fm
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams.update({
    'axes.unicode_minus': False,
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# 컬러
C_PURPLE = '#6C5CE7'
C_TEAL   = '#00B894'
C_CORAL  = '#E17055'
C_BLUE   = '#0984E3'
C_DARK   = '#2D3436'
C_GRAY   = '#B2BEC3'

# 데이터 다운로드
!kaggle datasets download -d cdc/national-health-and-nutrition-examination-survey --unzip -q

csvs = sorted(glob.glob('*.csv'))
print('다운로드된 파일:', csvs)


## STEP 2. 데이터 로드

In [ ]:

# Kaggle 미러 파일명은 고정되어 있음
demo = pd.read_csv('demographic.csv', encoding='latin-1')
quest = pd.read_csv('questionnaire.csv', encoding='latin-1')
exam = pd.read_csv('examination.csv', encoding='latin-1')

print('demographic:', demo.shape)
print('questionnaire:', quest.shape)
print('examination:', exam.shape)

# 필요한 변수만 미리 추출
phq_items = ['DPQ010','DPQ020','DPQ030','DPQ040','DPQ050','DPQ060','DPQ070','DPQ080','DPQ090']
quest_vars = ['SEQN'] + phq_items + ['PAD680', 'SLD010H', 'SMQ020', 'ALQ130']
exam_vars = ['SEQN', 'BMXWAIST', 'BMXHT', 'BMXBMI']
demo_vars = ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'INDFMPIR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA']

missing_demo = [c for c in demo_vars if c not in demo.columns]
missing_quest = [c for c in quest_vars if c not in quest.columns]
missing_exam = [c for c in exam_vars if c not in exam.columns]

print('누락 demo 변수:', missing_demo)
print('누락 quest 변수:', missing_quest)
print('누락 exam 변수:', missing_exam)

raw = (
    demo[demo_vars]
    .merge(quest[[c for c in quest_vars if c in quest.columns]], on='SEQN', how='inner')
    .merge(exam[[c for c in exam_vars if c in exam.columns]], on='SEQN', how='inner')
)

print(f'병합 결과: {raw.shape[0]:,}명 × {raw.shape[1]}개 변수')
raw.head(3)


## STEP 3. 전처리 + 파생변수

In [ ]:

df = raw.copy()

# ---- 유틸 함수 ----
def to_num(s):
    return pd.to_numeric(s, errors='coerce')

def replace_special_missing(series, bad_values):
    s = to_num(series).copy()
    s = s.replace(bad_values, np.nan)
    return s

def winsorize(s, q=0.01):
    lo, hi = s.quantile(q), s.quantile(1-q)
    return s.clip(lo, hi)

# ---- PHQ-9 ----
for c in phq_items:
    df[c] = replace_special_missing(df[c], [7, 9])

# 7문항 이상 응답 시 비례 환산 총점
valid_n = df[phq_items].notna().sum(axis=1)
raw_sum = df[phq_items].sum(axis=1, min_count=7)
df['PHQ9_total'] = np.where(valid_n >= 7, raw_sum * 9 / valid_n, np.nan)
df['PHQ9_total'] = df['PHQ9_total'].round(2)
df['dep_risk'] = (df['PHQ9_total'] >= 10).astype('float')

# ---- 좌식시간 ----
df['sedentary_min_day'] = replace_special_missing(df['PAD680'], [7777, 9999])
df['sedentary_min_day'] = df['sedentary_min_day'].where(
    (df['sedentary_min_day'] >= 0) & (df['sedentary_min_day'] <= 1440), np.nan
)

# ---- 수면시간 ----
df['sleep_hour'] = replace_special_missing(df['SLD010H'], [77, 99])
df['sleep_hour'] = df['sleep_hour'].where((df['sleep_hour'] >= 2) & (df['sleep_hour'] <= 12), np.nan)

def sleep_group(x):
    if pd.isna(x):
        return np.nan
    if x < 7:
        return '짧은 수면(<7h)'
    elif x <= 9:
        return '적정 수면(7-9h)'
    return '긴 수면(>9h)'

df['sleep_group'] = df['sleep_hour'].apply(sleep_group)

# ---- 중심성 비만 ----
df['waist_cm'] = to_num(df['BMXWAIST'])
df['height_cm'] = to_num(df['BMXHT'])
df['BMI'] = to_num(df['BMXBMI'])

df['waist_cm'] = df['waist_cm'].where((df['waist_cm'] >= 40) & (df['waist_cm'] <= 200), np.nan)
df['height_cm'] = df['height_cm'].where((df['height_cm'] >= 120) & (df['height_cm'] <= 220), np.nan)
df['BMI'] = df['BMI'].where((df['BMI'] >= 10) & (df['BMI'] <= 80), np.nan)

df['WHtR'] = df['waist_cm'] / df['height_cm']
df['central_obesity'] = (df['WHtR'] >= 0.50).astype('float')

# ---- 공변량 ----
df['age'] = to_num(df['RIDAGEYR'])
df['sex'] = to_num(df['RIAGENDR'])              # 1=male, 2=female
df['race'] = to_num(df['RIDRETH3'])
df['pir'] = to_num(df['INDFMPIR'])              # poverty income ratio
df['smoke_now'] = replace_special_missing(df['SMQ020'], [7, 9])  # ever smoked 100 cigs
df['alcohol_freq'] = replace_special_missing(df['ALQ130'], [777, 999]) if 'ALQ130' in df.columns else np.nan
df['weight'] = to_num(df['WTMEC2YR'])
df['psu'] = to_num(df['SDMVPSU'])
df['strata'] = to_num(df['SDMVSTRA'])

# ---- 성인 + 핵심 변수 결측 제거 ----
analysis_cols = [
    'PHQ9_total', 'dep_risk', 'sedentary_min_day', 'sleep_hour',
    'waist_cm', 'height_cm', 'BMI', 'WHtR', 'central_obesity',
    'age', 'sex', 'race', 'pir', 'smoke_now', 'weight', 'psu', 'strata'
]
df = df[df['age'] >= 20].copy()
df_an = df[analysis_cols].dropna().copy()

# ---- winsorize ----
for col in ['sedentary_min_day', 'WHtR', 'BMI', 'waist_cm']:
    df_an[col] = winsorize(df_an[col], q=0.01)

# ---- 범주화 ----
def dep_group(x):
    if x <= 4:
        return '없음(0-4)'
    elif x <= 9:
        return '경미(5-9)'
    elif x <= 14:
        return '중등도(10-14)'
    elif x <= 19:
        return '중등-중증(15-19)'
    return '중증(20+)'

df_an['dep_group'] = df_an['PHQ9_total'].apply(dep_group)
df_an['sex_label'] = df_an['sex'].map({1:'남성', 2:'여성'})
df_an['sedentary_q'] = pd.qcut(df_an['sedentary_min_day'], 4, labels=['Q1','Q2','Q3','Q4'])
df_an['sleep_short'] = (df_an['sleep_hour'] < 7).astype(int)
df_an['sleep_long'] = (df_an['sleep_hour'] > 9).astype(int)

print(f'✅ 최종 분석 대상: {len(df_an):,}명')
print(df_an[['PHQ9_total','sedentary_min_day','sleep_hour','WHtR','BMI']].describe().round(2))
print('\n우울군 분포')
print(df_an['dep_group'].value_counts())


## STEP 4. 기술통계

In [ ]:

# 가중 평균 유틸
def wmean(x, w):
    return np.sum(x * w) / np.sum(w)

summary = df_an.groupby('dep_group').apply(
    lambda d: pd.Series({
        'N': len(d),
        '가중 PHQ9 평균': wmean(d['PHQ9_total'], d['weight']),
        '가중 좌식시간 평균(분/일)': wmean(d['sedentary_min_day'], d['weight']),
        '가중 수면시간 평균(시간/일)': wmean(d['sleep_hour'], d['weight']),
        '가중 WHtR 평균': wmean(d['WHtR'], d['weight']),
        '가중 BMI 평균': wmean(d['BMI'], d['weight']),
        '가중 중심성비만 비율(%)': wmean(d['central_obesity'], d['weight']) * 100,
    })
).round(2)

summary




```
# 코드로 형식 지정됨
```

## STEP 5. 가중 회귀분석

In [ ]:

# Python에서는 NHANES 복합표본 분산추정을 완벽히 재현하기 어려워,
# 가중치 + HC3 robust SE 기반의 실용형 모델을 사용합니다.

covars = 'age + C(sex) + C(race) + pir + smoke_now'

# 1) 총연관: 우울감 -> 중심성 비만(연속형 WHtR)
m_total = smf.wls(
    f'WHtR ~ PHQ9_total + {covars}',
    data=df_an, weights=df_an['weight']
).fit(cov_type='HC3')

# 2) 우울감 -> 좌식시간
m_sed = smf.wls(
    f'sedentary_min_day ~ PHQ9_total + {covars}',
    data=df_an, weights=df_an['weight']
).fit(cov_type='HC3')

# 3) 우울감 -> 수면시간
m_sleep = smf.wls(
    f'sleep_hour ~ PHQ9_total + {covars}',
    data=df_an, weights=df_an['weight']
).fit(cov_type='HC3')

# 4) 우울감 + 좌식시간 + 수면 -> 중심성 비만(연속형)
m_joint = smf.wls(
    f'WHtR ~ PHQ9_total + sedentary_min_day + sleep_hour + {covars}',
    data=df_an, weights=df_an['weight']
).fit(cov_type='HC3')

# 5) 이분형 결과: 중심성 비만 여부
m_logit = smf.glm(
    f'central_obesity ~ PHQ9_total + sedentary_min_day + sleep_hour + {covars}',
    data=df_an,
    family=sm.families.Binomial(),
    freq_weights=df_an['weight']
).fit(cov_type='HC3')

print('총연관 모델')
display(m_total.summary2().tables[1].round(4))

print('좌식시간 모델')
display(m_sed.summary2().tables[1].round(4))

print('수면시간 모델')
display(m_sleep.summary2().tables[1].round(4))

print('공동 모델 (WHtR)')
display(m_joint.summary2().tables[1].round(4))

print('공동 모델 (중심성 비만 odds)')
display(pd.DataFrame({
    'coef': m_logit.params,
    'OR': np.exp(m_logit.params),
    'p': m_logit.pvalues
}).round(4))



```
# 코드로 형식 지정됨
```

## STEP 6. 매개효과 분석

In [ ]:

from joblib import Parallel, delayed
import numpy as np
import statsmodels.formula.api as smf

covars = 'age + C(sex) + C(race) + pir + smoke_now'

def _one_boot(df, mediator, covars, sample_frac, seed):
    """단일 부트스트랩 반복 - 워커 프로세스에서 실행"""
    rng = np.random.default_rng(seed)
    samp = df.sample(frac=sample_frac, replace=True,
                     random_state=int(rng.integers(1, 1_000_000)))
    try:
        ma = smf.wls(
            f'{mediator} ~ PHQ9_total + {covars}',
            data=samp, weights=samp['weight']
        ).fit()
        mb = smf.wls(
            f'WHtR ~ PHQ9_total + {mediator} + {covars}',
            data=samp, weights=samp['weight']
        ).fit()
        return ma.params['PHQ9_total'] * mb.params[mediator]
    except Exception:
        return np.nan

def bootstrap_indirect_parallel(data, mediator, n_boot=1000,
                                 subsample_ratio=0.5, seed=42, n_jobs=-1):
    """
    병렬 부트스트랩 간접효과 추정
    subsample_ratio: 0.5 → 절반 크기 샘플로 속도 2배 향상
    n_jobs=-1: 가용 코어 전부 사용
    """
    rng = np.random.default_rng(seed)
    seeds = rng.integers(1, 1_000_000, size=n_boot)

    ab = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_one_boot)(data, mediator, covars, subsample_ratio, s)
        for s in seeds
    )
    ab = np.array([v for v in ab if not np.isnan(v)])

    # subsample correction: var(ab)를 1/ratio 배 보정
    correction = np.sqrt(1 / subsample_ratio)
    mean_ab = np.mean(ab)
    spread = ab - mean_ab
    ab_corrected = mean_ab + spread * correction

    return {
        'indirect': mean_ab,
        'ci2.5':   float(np.percentile(ab_corrected, 2.5)),
        'ci97.5':  float(np.percentile(ab_corrected, 97.5)),
        'boot_values': ab_corrected,
        'n_valid': len(ab),
    }

# point estimates (기존과 동일)
a_sed   = m_sed.params['PHQ9_total']
b_sed   = m_joint.params['sedentary_min_day']
ind_sed = a_sed * b_sed

a_sleep   = m_sleep.params['PHQ9_total']
b_sleep   = m_joint.params['sleep_hour']
ind_sleep = a_sleep * b_sleep

print('⏳ 부트스트랩 시작...')
boot_sed   = bootstrap_indirect_parallel(df_an, 'sedentary_min_day',
                                          n_boot=1000, subsample_ratio=0.5,
                                          seed=42, n_jobs=-1)
boot_sleep = bootstrap_indirect_parallel(df_an, 'sleep_hour',
                                          n_boot=1000, subsample_ratio=0.5,
                                          seed=43, n_jobs=-1)
print(f'✅ 완료 | 좌식 유효 부트: {boot_sed["n_valid"]} | 수면 유효 부트: {boot_sleep["n_valid"]}')

med_results = pd.DataFrame([
    ['좌식시간', ind_sed,   boot_sed['ci2.5'],   boot_sed['ci97.5']],
    ['수면시간', ind_sleep, boot_sleep['ci2.5'], boot_sleep['ci97.5']],
], columns=['매개변수', '간접효과(a*b)', 'CI 2.5%', 'CI 97.5%'])

med_results['유의여부'] = np.where(
    (med_results['CI 2.5%'] > 0) | (med_results['CI 97.5%'] < 0),
    '유의', '비유의'
)
med_results.round(4)


## STEP 7. 성별 하위분석

In [ ]:

sex_results = []
for sex_value, sex_label in [(1, '남성'), (2, '여성')]:
    sub = df_an[df_an['sex'] == sex_value].copy()
    if len(sub) < 200:
        continue

    model = smf.wls(
        'WHtR ~ PHQ9_total + sedentary_min_day + sleep_hour + age + C(race) + pir + smoke_now',
        data=sub, weights=sub['weight']
    ).fit(cov_type='HC3')

    sex_results.append({
        '성별': sex_label,
        'N': len(sub),
        'PHQ9 β': model.params.get('PHQ9_total', np.nan),
        'PHQ9 p': model.pvalues.get('PHQ9_total', np.nan),
        '좌식시간 β': model.params.get('sedentary_min_day', np.nan),
        '좌식시간 p': model.pvalues.get('sedentary_min_day', np.nan),
        '수면시간 β': model.params.get('sleep_hour', np.nan),
        '수면시간 p': model.pvalues.get('sleep_hour', np.nan),
    })

sex_df = pd.DataFrame(sex_results).round(4)
sex_df


## STEP 8. 시각화

In [ ]:

os.makedirs('outputs', exist_ok=True)

# 1) 개념도
fig, ax = plt.subplots(figsize=(11, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.axis('off')

box_kw = dict(boxstyle='round,pad=0.5', linewidth=1.5)
ax.text(1.5, 3.0, '우울감\n(PHQ-9)', ha='center', va='center', fontsize=13,
        fontweight='bold', bbox=dict(**box_kw, facecolor='#E8DAFF', edgecolor=C_PURPLE))
ax.text(5.0, 4.6, '좌식시간\n(PAD680)', ha='center', va='center', fontsize=12,
        fontweight='bold', bbox=dict(**box_kw, facecolor='#D5F5E3', edgecolor=C_TEAL))
ax.text(5.0, 1.4, '수면시간\n(SLD010H)', ha='center', va='center', fontsize=12,
        fontweight='bold', bbox=dict(**box_kw, facecolor='#D6EAF8', edgecolor=C_BLUE))
ax.text(8.5, 3.0, '복부비만\n(WHtR, WHtR≥0.5)', ha='center', va='center', fontsize=13,
        fontweight='bold', bbox=dict(**box_kw, facecolor='#FDEBD0', edgecolor=C_CORAL))

ax.annotate('', xy=(4.1, 4.2), xytext=(2.2, 3.4), arrowprops=dict(arrowstyle='->', color=C_TEAL, lw=2))
ax.annotate('', xy=(4.1, 1.8), xytext=(2.2, 2.6), arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=2))
ax.annotate('', xy=(7.6, 3.4), xytext=(5.9, 4.2), arrowprops=dict(arrowstyle='->', color=C_TEAL, lw=2))
ax.annotate('', xy=(7.6, 2.6), xytext=(5.9, 1.8), arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=2))
ax.annotate('', xy=(7.6, 3.0), xytext=(2.3, 3.0), arrowprops=dict(arrowstyle='->', color=C_CORAL, lw=1.5, linestyle='dashed'))

ax.text(5.0, 5.4, f'a₁={a_sed:.3f}', ha='center', color=C_TEAL, fontweight='bold')
ax.text(5.0, 0.6, f'a₂={a_sleep:.3f}', ha='center', color=C_BLUE, fontweight='bold')
ax.text(6.9, 4.4, f'b₁={b_sed:.4f}', ha='center', color=C_TEAL, fontweight='bold')
ax.text(6.9, 1.5, f'b₂={b_sleep:.4f}', ha='center', color=C_BLUE, fontweight='bold')
ax.text(5.0, 2.55, f"c'={m_joint.params['PHQ9_total']:.4f}", ha='center', color=C_CORAL, fontweight='bold')

ax.set_title('개념 경로도: 우울감 → 좌식시간/수면 → 복부비만', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plot1_concept_path.png', dpi=150, bbox_inches='tight')
plt.show()

# 2) 우울군별 좌식시간/수면/WHtR
order = ['없음(0-4)', '경미(5-9)', '중등도(10-14)', '중등-중증(15-19)', '중증(20+)']
order = [o for o in order if o in df_an['dep_group'].unique()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, y, title, color in zip(
    axes,
    ['sedentary_min_day', 'sleep_hour', 'WHtR'],
    ['좌식시간(분/일)', '수면시간(시간/일)', 'WHtR'],
    [C_TEAL, C_BLUE, C_CORAL]
):
    sns.violinplot(data=df_an, x='dep_group', y=y, order=order, inner='box', cut=0, color=color, ax=ax)
    ax.set_xlabel('우울군')
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=20)
    ax.set_title(f'우울군별 {title}')
plt.tight_layout()
plt.savefig('outputs/plot2_dep_group_violin.png', dpi=150, bbox_inches='tight')
plt.show()

# 3) 산점도
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sample = df_an.sample(min(2000, len(df_an)), random_state=42)

pairs = [
    ('PHQ9_total', 'sedentary_min_day', '우울감 → 좌식시간', C_TEAL),
    ('PHQ9_total', 'sleep_hour', '우울감 → 수면시간', C_BLUE),
    ('PHQ9_total', 'WHtR', '우울감 → 복부비만', C_CORAL)
]
for ax, (x, y, title, color) in zip(axes, pairs):
    sns.regplot(data=sample, x=x, y=y, scatter_kws={'s':10, 'alpha':0.2, 'color':color},
                line_kws={'color':C_DARK, 'lw':2}, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.savefig('outputs/plot3_scatter_regression.png', dpi=150, bbox_inches='tight')
plt.show()

# 4) 부트스트랩 간접효과 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, boot_obj, title, color in zip(
    axes,
    [boot_sed, boot_sleep],
    ['좌식시간 경로 간접효과', '수면시간 경로 간접효과'],
    [C_TEAL, C_BLUE]
):
    vals = boot_obj['boot_values']
    ax.hist(vals, bins=60, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(np.mean(vals), color=C_DARK, lw=2, label=f'평균={np.mean(vals):.4f}')
    ax.axvline(boot_obj['ci2.5'], color=C_CORAL, ls='--', lw=2, label=f"2.5%={boot_obj['ci2.5']:.4f}")
    ax.axvline(boot_obj['ci97.5'], color=C_CORAL, ls='--', lw=2, label=f"97.5%={boot_obj['ci97.5']:.4f}")
    ax.axvline(0, color='gray', ls=':', lw=1.3)
    ax.set_title(title)
    ax.set_xlabel('간접효과')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('outputs/plot4_bootstrap_indirect.png', dpi=150, bbox_inches='tight')
plt.show()

# 5) 성별 하위군 forest-like plot
if not sex_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    y = np.arange(len(sex_df))
    ax.axvline(0, color='gray', lw=1)
    ax.scatter(sex_df['PHQ9 β'], y, s=80, color=C_CORAL)
    for i, row in sex_df.iterrows():
        ax.text(row['PHQ9 β'] + 0.0005, y[i], f"p={row['PHQ9 p']:.3f}", va='center', fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(sex_df['성별'])
    ax.set_xlabel('PHQ9 → WHtR β')
    ax.set_title('성별 하위군: 우울감과 WHtR의 연관성')
    plt.tight_layout()
    plt.savefig('outputs/plot5_sex_subgroup.png', dpi=150, bbox_inches='tight')
    plt.show()


## STEP 9. 결과 저장

In [ ]:

# 표 저장
summary.to_csv('outputs/table1_weighted_summary_by_dep_group.csv', encoding='utf-8-sig')
med_results.to_csv('outputs/table2_mediation_results.csv', index=False, encoding='utf-8-sig')
sex_df.to_csv('outputs/table3_sex_subgroup_results.csv', index=False, encoding='utf-8-sig')

# 회귀표 저장
def save_model_table(model, filename, logistic=False):
    if logistic:
        out = pd.DataFrame({
            'coef': model.params,
            'OR': np.exp(model.params),
            'p': model.pvalues
        })
    else:
        out = model.summary2().tables[1]
    out.to_csv(filename, encoding='utf-8-sig')

save_model_table(m_total, 'outputs/model_total_wls.csv')
save_model_table(m_sed, 'outputs/model_sedentary_wls.csv')
save_model_table(m_sleep, 'outputs/model_sleep_wls.csv')
save_model_table(m_joint, 'outputs/model_joint_wls.csv')
save_model_table(m_logit, 'outputs/model_central_obesity_glm.csv', logistic=True)

df_an.to_csv('outputs/analysis_dataset.csv', index=False, encoding='utf-8-sig')

print('✅ 저장 완료')
for f in sorted(glob.glob('outputs/*')):
    print(' -', f)



## 결과 해석 팁

이 노트북에서 가장 중요한 해석 포인트는 아래입니다.

1. `PHQ9_total → WHtR`의 총연관이 있는가  
2. `PHQ9_total → sedentary_min_day`, `PHQ9_total → sleep_hour`의 연관이 있는가  
3. 좌식시간/수면시간을 함께 넣었을 때 `PHQ9_total`의 계수가 약화되는가  
4. bootstrap 간접효과 95% CI가 0을 포함하는가  
5. BMI보다 **WHtR/복부비만**에서 패턴이 더 강한가  
6. 남녀에서 차이가 있는가

> 간접효과 CI가 0을 포함하면 **매개효과 비유의**로 해석해야 하며,  
> 이 경우 “완전 매개” 같은 표현은 쓰지 않는 것이 좋습니다.
